# 02 — Screen data-center relevance and score supportiveness with Dartmouth Chat Qwen 3.5

This notebook consumes `candidate_bills_collected.csv`. It first screens whether each bill materially affects physical commercial/industrial data centers, rejects obvious keyword false positives, and then gives relevant bills the recovered 1–10 supportiveness score.

It writes two resumable CSVs while preserving every collection and decision-tree column:

- `candidate_bills_data_center_relevant_scored.csv`
- `candidate_bills_not_data_center_relevant.csv`

Bills marked `uncertain` are retained in the first CSV for manual review but are not scored until their relevance is resolved. Opening this notebook makes no API calls.


## Security and model selection

Rotate the Dartmouth key embedded in the older notebooks before using this notebook. The key below is requested with `getpass`, stays only in kernel memory, and is not saved in notebook JSON.

The setup cell queries Dartmouth's live model list only when you run it, keeps model IDs containing both `qwen` and `3.5`, and asks you to choose if more than one matches. Environment-variable loading is intentionally not enabled. If you later adopt it, set `DARTMOUTH_API_KEY` outside Jupyter and replace the `getpass` line with `os.environ["DARTMOUTH_API_KEY"]` after ensuring no secret-bearing file is committed or shared.


In [9]:
# Run once if these packages are not already installed in the Python 3.13.9 kernel.
%pip install -q pandas requests openai


Note: you may need to restart the kernel to use updated packages.


In [10]:
import json
import re
import time
from getpass import getpass
from hashlib import sha256
from pathlib import Path

import pandas as pd
import requests
from openai import OpenAI


# -----------------------------
# Editable run settings
# -----------------------------
INPUT_CSV = Path.cwd() / "candidate_bills_collected.csv"
TEST_MODE = False          # Keep True for a five-bill dry run; then change to False.
TEST_BILL_COUNT = 5
RELEVANCE_BATCH_SIZE = 8
SCORING_BATCH_SIZE = 3
PAUSE_BETWEEN_BATCHES = 0.75
MAX_SOURCE_CHARS = 16_000

suffix = "_test" if TEST_MODE else ""
RELEVANT_CSV = Path.cwd() / f"candidate_bills_data_center_relevant_scored{suffix}.csv"
REJECTED_CSV = Path.cwd() / f"candidate_bills_not_data_center_relevant{suffix}.csv"

RELEVANCE_PROMPT_VERSION = "data-center-relevance-full-source-v2"
SUPPORTIVENESS_PROMPT_VERSION = "data-center-supportiveness-v1"

print("Input:", INPUT_CSV.resolve())
print("Relevant/scored:", RELEVANT_CSV.resolve())
print("Rejected:", REJECTED_CSV.resolve())
print("TEST MODE" if TEST_MODE else "FULL RUN")


Input: /Users/michael/Desktop/QSS20/Final Project/candidate_bills_collected.csv
Relevant/scored: /Users/michael/Desktop/QSS20/Final Project/candidate_bills_data_center_relevant_scored.csv
Rejected: /Users/michael/Desktop/QSS20/Final Project/candidate_bills_not_data_center_relevant.csv
FULL RUN


In [11]:
if not INPUT_CSV.exists():
    raise FileNotFoundError(
        f"Run notebook 01 first or edit INPUT_CSV. Missing: {INPUT_CSV.resolve()}"
    )

bills = pd.read_csv(INPUT_CSV, dtype=str).fillna("")
required = {"bill_id", "title", "abstract", "full_text"}
missing = required - set(bills.columns)
if missing:
    raise ValueError(f"Input is missing required columns: {sorted(missing)}")
if bills["bill_id"].duplicated().any():
    raise ValueError("Input contains duplicate bill_id values.")
if TEST_MODE:
    bills = bills.head(TEST_BILL_COUNT).copy()
print(f"Loaded {len(bills):,} bills.")


Loaded 1,094 bills.


In [12]:
# This cell makes one Dartmouth request to list models, but performs no classification.
DARTMOUTH_API_KEY = getpass.getpass("Rotated Dartmouth Chat API key: ").strip()
if not DARTMOUTH_API_KEY:
    raise ValueError("A Dartmouth Chat API key is required.")

model_response = requests.get(
    "https://chat.dartmouth.edu/api/models",
    headers={"Authorization": f"bearer {DARTMOUTH_API_KEY}"},
    timeout=60,
)
model_response.raise_for_status()
all_models = [item["id"] for item in model_response.json()["data"]]
qwen_35_models = [
    model_id for model_id in all_models
    if "qwen" in model_id.lower() and "3.5" in model_id.lower()
]
if not qwen_35_models:
    raise RuntimeError("Dartmouth returned no model ID containing both 'qwen' and '3.5'.")
for index, model_id in enumerate(qwen_35_models):
    print(f"[{index}] {model_id}")
selected_index = 0 if len(qwen_35_models) == 1 else int(input("Model number: "))
if not 0 <= selected_index < len(qwen_35_models):
    raise ValueError("Model selection is outside the available range.")
MODEL_ID = qwen_35_models[selected_index]
client = OpenAI(
    base_url="https://chat.dartmouth.edu/api",
    api_key=DARTMOUTH_API_KEY,
)
print("Selected:", MODEL_ID)


Rotated Dartmouth Chat API key:  ········


[0] qwen.qwen3.5-122b
Selected: qwen.qwen3.5-122b


In [13]:
RELEVANCE_SYSTEM_PROMPT = r'''
You are screening state legislation for an academic study of commercial and industrial data centers. Use only the supplied bill material.

A bill is relevant if it materially governs physical data-center development or operation, including explicit data-center facilities and technology-neutral unusually-large-load policies that plausibly cover data centers through tariffs, grid connection, cost allocation, generation obligations, water, siting, incentives, environmental permitting, construction, or workforce rules.

Reject false positives about government IT, privacy, records, health-data repositories, ordinary databases, broadband alone, or policies obviously targeted at another industry or ordinary scale. A generic mention of megawatts, energy generation, transmission, renewable energy, or interconnection is insufficient. If the source does not support a reliable decision, return uncertain.

Return only a JSON array, exactly one object per supplied bill_id:
{
  "bill_id": "supplied ID",
  "relevant": "yes, no, or uncertain",
  "relevance_type": "direct data-center reference, technology-neutral large-load policy, false positive, or uncertain",
  "policy_categories": ["tax incentives, siting/permitting, utility rates/cost allocation, grid/interconnection, energy requirements, water/environment, reporting/transparency, local authority, workforce/construction, moratorium/restrictions, study/commission, other, or not applicable"],
  "policy_direction": "supportive, conditional/regulatory, restrictive, neutral/study, mixed, not applicable, or uncertain",
  "confidence": 0.0,
  "reason": "no more than 45 words",
  "evidence_quote": "short exact quote from supplied source, or blank if unavailable"
}
'''.strip()

SUPPORTIVENESS_SYSTEM_PROMPT = r'''
You are assigning an ordinal policy-supportiveness score to state legislation affecting commercial and industrial data centers. Use the supplied bill source, relevance decision, policy classification, and evidence.

Score each bill from 1 through 10:
1 = prohibits data-center development or imposes a strong moratorium
2 = creates severe restrictions or major new barriers
3 = substantially discourages development
4 = creates modest net burdens or disincentives
5 = neutral, study-only, evenly mixed, irrelevant, or no clear effect
6 = modestly supportive but includes meaningful conditions
7 = clearly facilitates development
8 = provides substantial incentives, infrastructure, or streamlined approval
9 = strongly promotes or subsidizes development
10 = exceptionally strong promotion, subsidy, preemption, or development rights

Regulatory requirements are not automatically anti-data-center; determine their likely net effect. If evidence is insufficient, assign 5 with low confidence.

Return only a JSON array, exactly one object per supplied bill_id:
{
  "bill_id": "supplied ID",
  "supportiveness_score": 5,
  "score_confidence": 0.5,
  "score_reason": "no more than 45 words",
  "score_evidence_quote": "short exact quote from the supplied source"
}
'''.strip()

ALLOWED_RELEVANCE = {"yes", "no", "uncertain"}
ALLOWED_RELEVANCE_TYPES = {
    "direct data-center reference", "technology-neutral large-load policy",
    "false positive", "uncertain",
}
ALLOWED_DIRECTIONS = {
    "supportive", "conditional/regulatory", "restrictive", "neutral/study",
    "mixed", "not applicable", "uncertain",
}
ALLOWED_CATEGORIES = {
    "tax incentives", "siting/permitting", "utility rates/cost allocation",
    "grid/interconnection", "energy requirements", "water/environment",
    "reporting/transparency", "local authority", "workforce/construction",
    "moratorium/restrictions", "study/commission", "other", "not applicable",
}


In [14]:
def clean_text(value):
    text = str(value or "").replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    return re.sub(r"\n{3,}", "\n\n", text).strip()


def count_words(value):
    return len(re.findall(r"\b[\w'-]+\b", clean_text(value)))


def compact_source(text, limit=MAX_SOURCE_CHARS):
    text = clean_text(text)
    if len(text) <= limit:
        return text
    terms = [
        "data center", "datacenter", "large load", "megawatt", "utility",
        "water", "zoning", "permit", "tax", "electric", "interconnection",
    ]
    windows = [text[: limit // 3], text[-limit // 6 :]]
    lower = text.lower()
    radius = 1_500
    for term in terms:
        start = 0
        for _ in range(4):
            location = lower.find(term, start)
            if location < 0:
                break
            windows.append(text[max(0, location - radius) : location + radius])
            start = location + len(term)
    compact = "\n\n[...SOURCE WINDOW...]\n\n".join(dict.fromkeys(windows))
    return compact[:limit]


def source_for_row(row):
    abstract = clean_text(row.get("abstract", ""))
    full_text = clean_text(row.get("full_text", ""))
    if count_words(abstract) < 100 and full_text:
        source_scope, source_text = "full_text", full_text
        source_url = clean_text(row.get("full_text_source_url", ""))
    else:
        source_scope, source_text = "abstract", abstract
        source_url = clean_text(row.get("openstates_url", ""))
    source_text = compact_source(source_text)
    fingerprint = sha256(source_text.encode("utf-8")).hexdigest()
    return source_scope, source_url, source_text, fingerprint


def extract_json_array(raw_text):
    cleaned = str(raw_text or "").strip()
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"\s*```$", "", cleaned)
    start = cleaned.find("[")
    end = cleaned.rfind("]")
    if start == -1 or end == -1 or end < start:
        raise ValueError("The model response did not contain a JSON array.")
    return json.loads(cleaned[start : end + 1])


def verified_quote(quote, source):
    quote = clean_text(quote)
    return bool(quote and quote.casefold() in clean_text(source).casefold())


def call_model(system_prompt, records, maximum_retries=3):
    expected_ids = [record["bill_id"] for record in records]
    user_prompt = (
        "Process the following bills according to the codebook. "
        "Return only JSON.\n\n"
        + json.dumps(records, ensure_ascii=False, indent=2)
    )
    last_error = None

    for attempt in range(1, maximum_retries + 1):
        try:
            completion = client.chat.completions.create(
                model=MODEL_ID,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                stream=False,
            )
            raw_text = completion.choices[0].message.content
            results = extract_json_array(raw_text)
            returned_ids = [str(item.get("bill_id", "")) for item in results]

            if len(returned_ids) != len(set(returned_ids)):
                raise ValueError("Qwen returned a duplicate bill_id.")
            if len(returned_ids) != len(expected_ids) or set(returned_ids) != set(expected_ids):
                missing = sorted(set(expected_ids) - set(returned_ids))
                extra = sorted(set(returned_ids) - set(expected_ids))
                raise ValueError(
                    f"Qwen returned the wrong bill_id set. Missing: {missing}; extra: {extra}"
                )

            returned_by_id = {str(item["bill_id"]): item for item in results}
            return [returned_by_id[bill_id] for bill_id in expected_ids]

        except Exception as error:
            last_error = error
            print(f"Attempt {attempt} failed: {error}")
            if attempt < maximum_retries:
                time.sleep(2 ** attempt)

    raise RuntimeError(
        f"Model call failed after {maximum_retries} attempts: {last_error}"
    )


In [15]:
RELEVANCE_COLUMNS = [
    "dc_relevant", "dc_relevance_type", "dc_policy_categories",
    "dc_policy_direction", "dc_relevance_confidence", "dc_relevance_reason",
    "dc_relevance_evidence_quote", "dc_relevance_evidence_verified",
    "dc_source_scope", "dc_source_url", "dc_source_sha256", "dc_model",
    "dc_relevance_prompt_version", "dc_relevance_status", "dc_relevance_error",
]
SCORE_COLUMNS = [
    "supportiveness_score", "supportiveness_confidence", "supportiveness_reason",
    "supportiveness_evidence_quote", "supportiveness_evidence_verified",
    "supportiveness_model", "supportiveness_prompt_version",
    "supportiveness_status", "supportiveness_error",
]


def load_previous_results():
    frames = []
    for path in [RELEVANT_CSV, REJECTED_CSV]:
        if path.exists():
            frames.append(pd.read_csv(path, dtype=str).fillna(""))
    if not frames:
        return pd.DataFrame(columns=["bill_id"] + RELEVANCE_COLUMNS + SCORE_COLUMNS)
    prior = pd.concat(frames, ignore_index=True)
    return prior.drop_duplicates("bill_id", keep="last")


def save_split(frame):
    rejected = frame[frame["dc_relevant"] == "no"].copy()
    retained = frame[frame["dc_relevant"] != "no"].copy()
    retained.to_csv(RELEVANT_CSV, index=False)
    rejected.to_csv(REJECTED_CSV, index=False)


working = bills.copy()
previous = load_previous_results()
available = [
    column for column in ["bill_id"] + RELEVANCE_COLUMNS + SCORE_COLUMNS
    if column in previous.columns
]
if len(previous):
    working = working.merge(previous[available], on="bill_id", how="left")
working = working.fillna("")
for column in RELEVANCE_COLUMNS + SCORE_COLUMNS:
    if column not in working:
        working[column] = ""

# Invalidate a cached relevance result if its source or prompt changed.
for index, row in working.iterrows():
    _, _, _, fingerprint = source_for_row(row)
    relevance_stale = (
        row.get("dc_source_sha256", "") != fingerprint
        or row.get("dc_relevance_prompt_version", "") != RELEVANCE_PROMPT_VERSION
        or (row.get("dc_model", "") and row.get("dc_model", "") != MODEL_ID)
    )
    score_stale = (
        row.get("supportiveness_prompt_version", "") != SUPPORTIVENESS_PROMPT_VERSION
        or (
            row.get("supportiveness_model", "")
            and row.get("supportiveness_model", "") != MODEL_ID
        )
    )
    if relevance_stale:
        for column in RELEVANCE_COLUMNS + SCORE_COLUMNS:
            working.at[index, column] = ""
    elif score_stale:
        for column in SCORE_COLUMNS:
            working.at[index, column] = ""

print(f"Ready: {len(working):,} bills")


Ready: 1,094 bills


## Stage 1: relevance screening

This is the first classification cell. It checkpoints after every batch. False positives go to the rejected CSV; `yes` and `uncertain` stay in the retained CSV.


In [16]:
pending = working.index[working["dc_relevance_status"] != "classified"].tolist()
print(f"Relevance already complete: {len(working) - len(pending):,}; remaining: {len(pending):,}")

for start in range(0, len(pending), RELEVANCE_BATCH_SIZE):
    indices = pending[start : start + RELEVANCE_BATCH_SIZE]
    records = []
    sources = {}
    for index in indices:
        row = working.loc[index]
        scope, url, source, fingerprint = source_for_row(row)
        sources[str(row["bill_id"])] = source
        records.append(
            {
                "bill_id": str(row["bill_id"]), "state": str(row.get("state", "")),
                "session": str(row.get("session", "")), "identifier": str(row.get("identifier", "")),
                "title": str(row.get("title", "")), "source_scope": scope,
                "source_text": source,
            }
        )
    try:
        results = call_model(RELEVANCE_SYSTEM_PROMPT, records)
        for index, result in zip(indices, results):
            bill_id = str(working.at[index, "bill_id"])
            scope, url, _, fingerprint = source_for_row(working.loc[index])
            relevant = str(result.get("relevant", "uncertain")).lower().strip()
            relevance_type = str(result.get("relevance_type", "uncertain")).lower().strip()
            direction = str(result.get("policy_direction", "uncertain")).lower().strip()
            categories = result.get("policy_categories", [])
            categories = categories if isinstance(categories, list) else [categories]
            categories = [str(c).lower().strip() for c in categories if str(c).lower().strip() in ALLOWED_CATEGORIES]
            try:
                confidence = min(1.0, max(0.0, float(result.get("confidence", 0))))
            except (TypeError, ValueError):
                confidence = 0.0
            quote = clean_text(result.get("evidence_quote", ""))
            working.at[index, "dc_relevant"] = relevant if relevant in ALLOWED_RELEVANCE else "uncertain"
            working.at[index, "dc_relevance_type"] = relevance_type if relevance_type in ALLOWED_RELEVANCE_TYPES else "uncertain"
            working.at[index, "dc_policy_categories"] = "|".join(categories or ["other"])
            working.at[index, "dc_policy_direction"] = direction if direction in ALLOWED_DIRECTIONS else "uncertain"
            working.at[index, "dc_relevance_confidence"] = confidence
            working.at[index, "dc_relevance_reason"] = clean_text(result.get("reason", ""))
            working.at[index, "dc_relevance_evidence_quote"] = quote
            working.at[index, "dc_relevance_evidence_verified"] = verified_quote(quote, sources[bill_id])
            working.at[index, "dc_source_scope"] = scope
            working.at[index, "dc_source_url"] = url
            working.at[index, "dc_source_sha256"] = fingerprint
            working.at[index, "dc_model"] = MODEL_ID
            working.at[index, "dc_relevance_prompt_version"] = RELEVANCE_PROMPT_VERSION
            working.at[index, "dc_relevance_status"] = "classified"
            working.at[index, "dc_relevance_error"] = ""
    except Exception as error:
        print("Batch error:", error)
        for index in indices:
            working.at[index, "dc_relevance_status"] = "error"
            working.at[index, "dc_relevance_error"] = str(error)
    save_split(working)
    time.sleep(PAUSE_BETWEEN_BATCHES)
    print(f"Batch {start} done")

print("Relevance stage complete.")
display(working["dc_relevant"].value_counts(dropna=False))


Relevance already complete: 17; remaining: 1,077
Batch 0 done
Batch 8 done
Batch 16 done
Batch 24 done
Batch 32 done
Batch 40 done
Batch 48 done
Batch 56 done
Batch 64 done
Batch 72 done
Batch 80 done
Batch 88 done
Batch 96 done
Batch 104 done
Batch 112 done
Batch 120 done
Batch 128 done
Batch 136 done
Batch 144 done
Batch 152 done
Batch 160 done
Batch 168 done
Batch 176 done
Batch 184 done
Batch 192 done
Attempt 1 failed: Qwen returned the wrong bill_id set. Missing: ['ocd-bill/c9c4f33b-fca0-4cb6-b9c8-3c3fcb633486']; extra: []
Batch 200 done
Batch 208 done
Batch 216 done
Batch 224 done
Batch 232 done
Batch 240 done
Batch 248 done
Batch 256 done
Batch 264 done
Batch 272 done
Batch 280 done
Batch 288 done
Batch 296 done
Batch 304 done
Batch 312 done
Batch 320 done
Batch 328 done
Batch 336 done
Batch 344 done
Batch 352 done
Batch 360 done
Batch 368 done
Batch 376 done
Batch 384 done
Batch 392 done
Batch 400 done
Attempt 1 failed: Connection error.
Attempt 2 failed: Connection error.
Atte

dc_relevant
no           839
yes          236
uncertain     11
               8
Name: count, dtype: int64

## Stage 2: 1–10 supportiveness scoring

Only bills classified `yes` are scored. Rejected bills remain auditable in their own CSV. Uncertain bills stay in the retained CSV with `supportiveness_status = manual_relevance_review_needed`.


In [21]:
uncertain_indices = working.index[working["dc_relevant"] == "uncertain"]
working.loc[uncertain_indices, "supportiveness_status"] = "manual_relevance_review_needed"

score_pending = working.index[
    (working["dc_relevant"] == "yes")
    & (working["supportiveness_status"] != "scored")
].tolist()
print(f"Bills remaining to score: {len(score_pending):,}")

for start in range(0, len(score_pending), SCORING_BATCH_SIZE):
    indices = score_pending[start : start + SCORING_BATCH_SIZE]
    records = []
    sources = {}
    for index in indices:
        row = working.loc[index]
        _, _, source, _ = source_for_row(row)
        sources[str(row["bill_id"])] = source
        records.append(
            {
                "bill_id": str(row["bill_id"]), "state": str(row.get("state", "")),
                "identifier": str(row.get("identifier", "")), "title": str(row.get("title", "")),
                "relevance_type": str(row.get("dc_relevance_type", "")),
                "policy_categories": str(row.get("dc_policy_categories", "")),
                "policy_direction": str(row.get("dc_policy_direction", "")),
                "relevance_reason": str(row.get("dc_relevance_reason", "")),
                "source_text": source,
            }
        )
    try:
        results = call_model(SUPPORTIVENESS_SYSTEM_PROMPT, records)
        for index, result in zip(indices, results):
            bill_id = str(working.at[index, "bill_id"])
            try:
                score = min(10, max(1, int(round(float(result.get("supportiveness_score", 5))))))
            except (TypeError, ValueError):
                score = 5
            try:
                confidence = min(1.0, max(0.0, float(result.get("score_confidence", 0))))
            except (TypeError, ValueError):
                confidence = 0.0
            quote = clean_text(result.get("score_evidence_quote", ""))
            working.at[index, "supportiveness_score"] = score
            working.at[index, "supportiveness_confidence"] = confidence
            working.at[index, "supportiveness_reason"] = clean_text(result.get("score_reason", ""))
            working.at[index, "supportiveness_evidence_quote"] = quote
            working.at[index, "supportiveness_evidence_verified"] = verified_quote(quote, sources[bill_id])
            working.at[index, "supportiveness_model"] = MODEL_ID
            working.at[index, "supportiveness_prompt_version"] = SUPPORTIVENESS_PROMPT_VERSION
            working.at[index, "supportiveness_status"] = "scored"
            working.at[index, "supportiveness_error"] = ""
    except Exception as error:
        print("Scoring batch error:", error)
        for index in indices:
            working.at[index, "supportiveness_status"] = "error"
            working.at[index, "supportiveness_error"] = str(error)
    save_split(working)
    time.sleep(PAUSE_BETWEEN_BATCHES)
    print(f"Batch {start} done")

save_split(working)
print("Scoring stage complete.")
print("Retained:", RELEVANT_CSV.resolve())
print("Rejected:", REJECTED_CSV.resolve())


Bills remaining to score: 62
Batch 0 done
Batch 3 done
Batch 6 done
Batch 9 done
Batch 12 done
Batch 15 done
Batch 18 done
Batch 21 done
Batch 24 done
Batch 27 done
Batch 30 done
Batch 33 done
Batch 36 done
Batch 39 done
Batch 42 done
Batch 45 done
Batch 48 done
Batch 51 done
Batch 54 done
Batch 57 done
Batch 60 done
Scoring stage complete.
Retained: /Users/michael/Desktop/QSS20/Final Project/candidate_bills_data_center_relevant_scored.csv
Rejected: /Users/michael/Desktop/QSS20/Final Project/candidate_bills_not_data_center_relevant.csv


In [22]:
# Local quality-control summary; no API calls.
retained = pd.read_csv(RELEVANT_CSV, dtype=str).fillna("")
rejected = pd.read_csv(REJECTED_CSV, dtype=str).fillna("")
print(f"Retained (yes + uncertain): {len(retained):,}")
print(f"Rejected: {len(rejected):,}")
display(retained["dc_relevant"].value_counts(dropna=False))
display(retained["supportiveness_score"].value_counts(dropna=False).sort_index())

review_queue = retained[
    (retained["dc_relevant"] == "uncertain")
    | (pd.to_numeric(retained["dc_relevance_confidence"], errors="coerce").fillna(0) < 0.70)
    | (retained["dc_relevance_evidence_verified"].astype(str).str.lower() != "true")
]
print(f"Manual relevance/evidence review queue: {len(review_queue):,}")
display(
    review_queue[
        ["state", "identifier", "title", "dc_relevant", "dc_relevance_confidence",
         "dc_relevance_reason", "dc_relevance_evidence_quote", "dc_source_url"]
    ].head(25)
)


Retained (yes + uncertain): 255
Rejected: 839


dc_relevant
yes          236
uncertain     11
               8
Name: count, dtype: int64

supportiveness_score
     19
1     4
2     7
3    25
4    76
5    34
6    18
7    38
8    32
9     2
Name: count, dtype: int64

Manual relevance/evidence review queue: 196


,state,identifier,title,dc_relevant,dc_relevance_confidence,dc_relevance_reason,dc_relevance_evidence_quote,dc_source_url
3,California,AB 93,Water resources: data centers.,yes,0.95,Explicitly regulates data center water use rep...,This bill would require a person who owns or o...,https://openstates.org/ca/bills/20252026/AB93/
4,California,AB 222,Data centers: power usage effectiveness: cost ...,yes,0.95,Mandates PUE reporting for data centers and re...,This bill would require the Energy Commission ...,https://openstates.org/ca/bills/20252026/AB222/
7,California,AB 1347,Electrical modernization zones.,uncertain,0.5,Bill establishes 'load growth priority areas' ...,identify six electrical infrastructure moderni...,https://openstates.org/ca/bills/20252026/AB1347/
8,California,AB 1408,Electricity: interconnections.,yes,0.9,Bill mandates utilities to evaluate and utiliz...,require each electrical corporation... to requ...,https://openstates.org/ca/bills/20252026/AB1408/
9,California,AB 1577,Data centers: reporting.,yes,0.95,Requires data center owners to submit location...,require the owner of a data center... to submi...,https://openstates.org/ca/bills/20252026/AB1577/
11,California,SB 978,Data centers: labor: electricity rates.,yes,0.95,Mandates special PUC rate structures for 75MW+...,require the PUC to establish a special rate st...,https://openstates.org/ca/bills/20252026/SB978/
13,California,AB 2383,Electricity: data centers.,yes,0.95,Requires PUC to create separate electricity co...,provide for a classification of retail electri...,https://openstates.org/ca/bills/20252026/AB2383/
14,California,AB 2469,Data centers: water use disclosures.,yes,0.95,Prohibits local agencies from approving data c...,"prohibit a city, county, or city and county fr...",https://openstates.org/ca/bills/20252026/AB2469/
15,California,AB 2619,Water resources: data centers.,yes,0.95,Requires data center owners to report water us...,require a person who owns or operates a data c...,https://openstates.org/ca/bills/20252026/AB2619/
16,Georgia,HB 101,Income tax; change certain definitions,yes,0.95,Explicitly addresses sales and use tax exempti...,high-technology data center customer that cont...,https://www.legis.ga.gov/api/legislation/docum...
